# M_02 — En-matching (Lei & Lei 2024) — implémentation fidèle

Appariement simultané **nœuds + arcs** entre graphes topologiques via le modèle ILP de
**Lei & Lei (2024)**, *ISPRS Int. J. Geo-Inf.*, doi:10.3390/ijgi13010015.

**Entrée** : `nodes.csv` + `edges.gpkg` produits par M_01 (EPSG:2154, mètres).  
**Sortie** : `match_nodes.csv`, `match_edges.csv` — correspondances pour M_03 (CRH).

---

## Formulation ILP complète (Section 3.2)

### Variables de décision
| Variable | Papier | Ici | Signification |
|---|---|---|---|
| Appariement d'arcs | $x_{ij} \in \{0,1\}$ | `x_arcs[(i,j)]` | arc $i \in I$ apparié à l'arc $j \in J$ |
| Appariement de nœuds | $u_{rs} \in \{0,1\}$ | `u_nodes[(r,s)]` | nœud $r \in V(I)$ apparié au nœud $s \in V(J)$ |

### Ensembles candidats
$$E = \{(i,j) \mid D(i,j) < c\} \quad N = \{(r,s) \mid d(r,s) < c\}$$

### Objectif (éq. 1)
$$\text{Maximiser } Z = \underbrace{\sum_{(i,j)\in E} (B - D_{ij} + \gamma B)\, x_{ij}}_{\text{terme arcs}} + \underbrace{\beta \sum_{(r,s)\in N} (B - d_{rs})\, u_{rs}}_{\text{terme nœuds } (\times \beta)}$$

- $B = \max(\max_{E} D_{ij},\, \max_{N} d_{rs}) + 1$ (garantit positivité des similarités)
- $\beta = 4$ : les nœuds sont pondérés plus que les arcs (1 nœud ancre ~4 arcs)
- $\gamma = 0.5$ : bonus de connectivité (nœuds de fort degré prioritaires)

### Contraintes (éqs. 2–11)
| Éq. | Contrainte | Rôle |
|---|---|---|
| (2)(3) | $\sum_j x_{ij} \leq 1$, $\sum_i x_{ij} \leq 1$ | Bijection des arcs |
| (4)(5) | $\sum_s u_{rs} \leq 1$, $\sum_r u_{rs} \leq 1$ | Bijection des nœuds |
| (6) | $u_{f(i)f(j)} + u_{f(i)t(j)} \geq x_{ij}$ | Départ de $i$ ↔ extrémité de $j$ |
| (7) | $u_{f(i)f(j)} + u_{t(i)f(j)} \geq x_{ij}$ | Départ de $j$ ↔ extrémité de $i$ |
| (8) | $u_{t(i)t(j)} + u_{f(i)t(j)} \geq x_{ij}$ | Arrivée de $i$ ↔ extrémité de $j$ |
| (9) | $u_{t(i)t(j)} + u_{t(i)f(j)} \geq x_{ij}$ | Arrivée de $j$ ↔ extrémité de $i$ |

Les éqs. (6)–(9) ensemble forcent la cohérence tête-à-tête ou tête-à-queue.


In [1]:
import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from itertools import combinations
import pulp
import time
import warnings
warnings.filterwarnings('ignore')


In [2]:
# ============================================================
#  CONFIGURATION — adapter selon le type de données
# ============================================================

# --- Chemins ---
MATCHING_DIR = r'C:/Users/rocrom/Resolving-Conflicts-in-Heterogeneous-Data--CRH_Framework/final'
SOURCES      = ['bdtopo', 'cadastre', 'geofla']

# ============================================================
#  Paramètres ILP (Lei & Lei 2024, Section 3.2)
# ============================================================
BETA  = 4.0   # β : poids des NŒUDS dans l'objectif
               # ► Justification : 1 nœud ancre β arcs adjacents
               # ► Frontières communales (degré 3) → β = 3
               # ► Réseaux routiers (degré moyen 4)  → β = 4  [valeur papier]
               # ► À recalibrer empiriquement si degré moyen connu

GAMMA = 0.5   # γ : bonus de connectivité pour nœuds de fort degré
               # ► γ = 0   : désactivé (risque de "connectivity gaps")
               # ► γ = 0.5 : valeur papier, recommandée
               # ► γ > 1.5 : dégradation observée (papier Fig. 5/6)

# ============================================================
#  Cut-off géométrique c (mètres)
# ============================================================
C_MANUAL = None   # None = calibration automatique (recommandé)
                  # Sinon, fixer explicitement :
                  #   ~ 10–30 m pour données cadastrales FR
                  #   ~ 100 m  pour TIGER/Line US (papier)
                  #   ~ 50 m   pour routes OSM vs IGN

C_MARGIN = 1.5    # c_auto = distance_max_observée × C_MARGIN (arrondi 10 m)

C_STRATEGY = 'per_pair'   # 'per_pair' : un cut-off propre à chaque paire de sources
                          #              (recommandé — robuste aux sources hétérogènes)
                          # 'global'   : un seul cut-off (max global) pour toutes les paires

CALIB_MIN_DEGREE = 3      # calibrer c uniquement sur les VRAIES jonctions (degré ≥ 3).
                          # Les nœuds de degré 1 sont des extrémités de bord (zone test
                          # tronquée) : leur position dépend de la découpe, pas de la
                          # géographie — les inclure pollue la calibration.
                          # Mettre 1 pour inclure tous les nœuds (déconseillé).
                          # Pour les routes : 3 reste pertinent (vraies intersections).

# ============================================================
#  Portée du matching — quels nœuds l'ILP a le droit d'apparier
# ============================================================
MATCH_SCOPE = 'junctions'  # 'junctions' (RIGOUREUX, défaut) : n'apparie que les vraies
                           #   jonctions topologiques (degré ≥ MATCH_MIN_DEGREE). Les nœuds
                           #   de bord (degré 1), artefacts de découpe de la zone test, sont
                           #   exclus — ils ne représentent pas le même objet entre sources.
                           #   C'est l'approche du papier (Lei & Lei matchent des jonctions).
                           #
                           # 'all' (DÉMONSTRATIF) : apparie TOUS les nœuds, bords inclus.
                           #   Utile pour une présentation (montre 4/4 au lieu de 1/1 sur la
                           #   zone test), mais les appariements de bord ne sont pas fiables
                           #   et ne doivent pas entrer dans les métriques de qualité.

MATCH_MIN_DEGREE = 3       # seuil de degré pour qu'un nœud soit une « jonction »
                           #   (utilisé seulement si MATCH_SCOPE == 'junctions').

# ============================================================
#  Candidature des arcs
# ============================================================
#
#  MODE FRONTIÈRES COMMUNALES (actuel)
#  ─────────────────────────────────────
#  Les arcs sont filtrés par identifiant sémantique (code INSEE).
#  La distance de Hausdorff est quand même calculée pour l'objectif.
#
#  MODE RÉSEAUX ROUTIERS (futur, D031 complet)
#  ─────────────────────────────────────────────
#  Désactiver USE_SEMANTIC → la candidature devient purement
#  géométrique (Hausdorff < c). Optionnel : ajouter distance
#  de Levenshtein sur les noms de rues (Section 4.2, papier).
#  Voir Section 4 du notebook pour adapter.

USE_SEMANTIC = True

def semantic_label(row):
    """Label sémantique d'un arc. Modifier selon le type de données.

    Communes : frozenset des codes INSEE des deux communes séparées par l'arc.
    Routes    : None  (désactiver USE_SEMANTIC).
    Bâtiments : frozenset des codes de parcelles.
    """
    return frozenset([str(row['commune_a']), str(row['commune_b'])])

# ============================================================
#  Résolution ILP
# ============================================================
SOLVER_TIME_LIMIT = 300   # secondes max par paire de sources
                           # ↑ pour grands réseaux (D031 complet)
SOLVER_VERBOSE    = False  # True → affiche progression du solver CBC


## 1. Chargement des données

In [3]:
nodes_df  = pd.read_csv(f'{MATCHING_DIR}/nodes.csv')
edges_gdf = gpd.read_file(f'{MATCHING_DIR}/edges.gpkg')

# Sous-tables par source
nodes = {src: nodes_df[nodes_df.source == src].reset_index(drop=True)
         for src in SOURCES}
edges = {src: edges_gdf[edges_gdf.source == src].reset_index(drop=True)
         for src in SOURCES}

# Vérification
for src in SOURCES:
    n, e = len(nodes[src]), len(edges[src])
    has_geom = 'geometry' in edges[src].columns
    print(f'{src:12s}: {n} nœuds  {e} arcs partagés  [géométrie: {has_geom}]')

print(f'\nCRS : {edges_gdf.crs}')
print('Colonnes arcs :', list(edges_gdf.columns))


bdtopo      : 1158 nœuds  1570 arcs partagés  [géométrie: True]
cadastre    : 1235 nœuds  1691 arcs partagés  [géométrie: True]
geofla      : 1157 nœuds  1565 arcs partagés  [géométrie: True]

CRS : EPSG:2154
Colonnes arcs : ['source', 'edge_id', 'node_start', 'node_end', 'n_vertices', 'commune_a', 'commune_b', 'geometry']


## 2. Utilitaires de distance

- **dist_nodes** : distance euclidienne entre nœuds (mètres, EPSG:2154)
- **hausdorff_dist** : distance de Hausdorff symétrique entre géométries d'arcs  
  (utilisée comme $D_{ij}$ dans l'objectif, cf. Section 2.1 du papier)
- **compute_B** : constante $B$ garantissant la positivité de toutes les similarités


In [4]:
def dist_nodes(xr, xs):
    """Distance euclidienne entre deux nœuds (arrays numpy 2D)."""
    return float(np.linalg.norm(xr - xs))


def hausdorff_dist(geom_a, geom_b):
    """Distance de Hausdorff symétrique entre deux géométries Shapely.
    
    Renvoie max(H(A→B), H(B→A)).
    Shapely's hausdorff_distance() calcule déjà le max des deux directions.
    """
    return float(geom_a.hausdorff_distance(geom_b))


def compute_B(d_rs_vals, D_ij_vals):
    """Constante B = max(max_d_rs, max_D_ij) + 1 (éq. après éq. 1 dans le papier).
    
    Garantit que B - d_rs > 0 et B - D_ij > 0 pour tous les candidats,
    ce qui assure que toutes les similarités dans l'objectif sont positives.
    """
    all_vals = list(d_rs_vals) + list(D_ij_vals)
    return (max(all_vals) if all_vals else 1.0) + 1.0


def node_coords(nodes_src):
    """Retourne un dict {node_id: np.array([x, y])}."""
    return {int(r.node_id): np.array([r.x, r.y]) for _, r in nodes_src.iterrows()}


def node_degrees(edges_src):
    """Retourne un dict {node_id: degré} depuis la table des arcs.
    
    Note : pour les frontières communales, seuls les arcs partagés sont présents,
    donc le degré mesuré ici est le degré DANS LE SOUS-GRAPHE partagé.
    Pour les réseaux routiers, edges doit contenir TOUS les arcs.
    """
    deg = {}
    for _, e in edges_src.iterrows():
        for nid in (int(e.node_start), int(e.node_end)):
            deg[nid] = deg.get(nid, 0) + 1
    return deg


## 3. Calibration du cut-off $c$

Le cut-off $c$ est le seuil géométrique au-delà duquel deux entités ne peuvent pas
être candidates à l'appariement (cf. Section 3.2 du papier, définition de $E$ et $N$).

**Stratégie :** si `USE_SEMANTIC = True`, on identifie les nœuds homologues via les
labels d'arcs (codes INSEE) et on mesure leurs distances réelles.

**Calibration sur les vraies jonctions uniquement (`CALIB_MIN_DEGREE`).**  
Tous les nœuds n'ont pas le même statut :

- les nœuds de **degré ≥ 3** sont de **vraies jonctions topologiques** (points triples,
  carrefours) — leur position a un sens géographique et reflète la précision des sources ;
- les nœuds de **degré 1** sont des **extrémités de bord** : ils marquent l'endroit où une
  frontière sort de la zone test découpée. Leur position dépend de **où chaque source pose
  son dernier sommet avant la coupe**, pas de la géographie. Deux sources peuvent placer
  ce point à des dizaines de mètres l'une de l'autre sans que ce soit une vraie divergence.

Calibrer $c$ sur les nœuds de bord gonfle donc artificiellement le seuil (c'est ce qui
donnait $c$ = 120 m à cause d'extrémités tronquées). On restreint la calibration aux
jonctions de degré ≥ `CALIB_MIN_DEGREE` (= 3 par défaut).

> **Important** : ce filtre ne concerne que **le calcul de la valeur** de $c$.
> Le cut-off, une fois fixé, s'applique à **tous** les nœuds dans l'ILP (bords inclus).

**Un $c$ par paire** (`C_STRATEGY = 'per_pair'`) reste recommandé : BD Topo/Cadastre
(haute résolution) et GÉOFLA (basse résolution) n'ont pas la même précision.

Si `USE_SEMANTIC = False` (routes, bâtiments), fixer `C_MANUAL` en mètres.


In [5]:
# ============================================================
#  M_02 — Calibration ROBUSTE du cut-off c  (avec plafond C_MAX)
#  c = min( median + k*MAD , C_MAX )
#  - median + k*MAD : robuste aux quelques aberrations
#  - C_MAX          : garde-fou absolu. Indispensable a l'echelle : quand la
#                     population d'artefacts grandit (Cadastre / rivieres), la MAD
#                     gonfle et la calibration derive (on a vu c=300-360 m). Au-dela
#                     de C_MAX, une jonction n'est pas un homologue : elle reste NON
#                     appariee au lieu d'etre forcee.
# ============================================================
import numpy as np

C_MAD_K = 5.0     # k de median + k*MAD
C_MAX   = 100   # m : plafond absolu sur le cut-off.
                  # Justif : 2 jonctions reellement homologues (IGN) sont a < ~70 m
                  # (GEOFLA generalise plafonnait a ~64 m sur les cas propres).


def homolog_pairs(eA, eB, cA, cB, semantic_fn, keep_nodes_A=None, keep_nodes_B=None):
    """Paires de noeuds homologues via labels d'arcs partages."""
    lab_A = {semantic_fn(row): row for _, row in eA.iterrows()}
    out = {}
    for _, eb in eB.iterrows():
        lbl = semantic_fn(eb)
        if lbl not in lab_A:
            continue
        ea = lab_A[lbl]
        a_s, a_e = int(ea.node_start), int(ea.node_end)
        b_s, b_e = int(eb.node_start), int(eb.node_end)
        xa_s, xa_e = cA[a_s], cA[a_e]
        xb_s, xb_e = cB[b_s], cB[b_e]
        d_dir = np.linalg.norm(xa_s - xb_s) + np.linalg.norm(xa_e - xb_e)
        d_rev = np.linalg.norm(xa_s - xb_e) + np.linalg.norm(xa_e - xb_s)
        if d_dir <= d_rev:
            cand = [((a_s, b_s), xa_s, xb_s), ((a_e, b_e), xa_e, xb_e)]
        else:
            cand = [((a_s, b_e), xa_s, xb_e), ((a_e, b_s), xa_e, xb_s)]
        for (na, nb_), pa, pb in cand:
            if keep_nodes_A is not None and na not in keep_nodes_A:
                continue
            if keep_nodes_B is not None and nb_ not in keep_nodes_B:
                continue
            out[(na, nb_)] = float(np.linalg.norm(pa - pb))
    return out


def round_up_10(v):
    return int(np.ceil(v / 10.0)) * 10


def junction_nodes(src, min_degree):
    deg = node_degrees(edges[src])
    return {nid for nid, d in deg.items() if d >= min_degree}


def c_mad(vals, k, c_max=None):
    """Cut-off robuste = median + k*MAD, plafonne a c_max. Renvoie (med, mad, c_brut, c_plafonne)."""
    a = np.array(vals, dtype=float)
    med = float(np.median(a))
    mad = float(np.median(np.abs(a - med)) * 1.4826)
    c_brut = med + k * mad
    c = min(c_brut, c_max) if c_max is not None else c_brut
    return med, mad, c_brut, c


def calibrate_c(edges_by_src, nodes_by_src, pairs, semantic_fn=None,
                mad_k=5.0, c_max=None, min_degree=3):
    """Calibre c par paire via min(median + k*MAD, C_MAX), sur les jonctions (degre >= min_degree)."""
    if semantic_fn is None:
        print('⚠  Calibration auto impossible sans filtre semantique. Fixer C_MANUAL.')
        return None

    jct = {src: junction_nodes(src, min_degree) for src in SOURCES}
    n_jct = {src: len(jct[src]) for src in SOURCES}
    print(f'Noeuds retenus pour la calibration (degre >= {min_degree}) : '
          + ', '.join(f'{s}={n_jct[s]}' for s in SOURCES))
    if all(v == 0 for v in n_jct.values()):
        print('⚠  Aucune jonction — repli sur tous les noeuds.')
        jct = None

    per_pair, detail, c_values = {}, {}, []
    for srcA, srcB in pairs:
        cA = node_coords(nodes_by_src[srcA]); cB = node_coords(nodes_by_src[srcB])
        keepA = jct[srcA] if jct is not None else None
        keepB = jct[srcB] if jct is not None else None
        pd_ = homolog_pairs(edges_by_src[srcA], edges_by_src[srcB], cA, cB,
                            semantic_fn, keep_nodes_A=keepA, keep_nodes_B=keepB)
        detail[(srcA, srcB)] = pd_
        if not pd_:
            per_pair[(srcA, srcB)] = {'median': None, 'mad': None, 'max': None, 'c': None,
                                      'c_brut': None, 'plafonne': False,
                                      'n_used': 0, 'n_out': 0, 'outliers': []}
            continue
        vals = list(pd_.values())
        med, mad, c_brut, c_raw = c_mad(vals, mad_k, c_max=c_max)
        c = round_up_10(c_raw)
        plafonne = (c_max is not None) and (c_brut > c_max)
        outliers = sorted((v for v in vals if v > c), reverse=True)
        per_pair[(srcA, srcB)] = {
            'median': med, 'mad': mad, 'max': float(max(vals)), 'c': c,
            'c_brut': round(c_brut, 1), 'plafonne': plafonne,
            'n_used': len(vals), 'n_out': len(outliers), 'outliers': outliers}
        c_values.append(c)

    c_global = max(c_values) if c_values else None

    # --- Affichage ---
    titre_max = f' (plafond C_MAX={c_max:.0f} m)' if c_max is not None else ''
    print(f'\n=== Calibration robuste (median + {mad_k}*MAD){titre_max}, jonctions deg >= {min_degree} ===\n')
    for (srcA, srcB), st in per_pair.items():
        if st['c'] is None:
            print(f'{srcA} ↔ {srcB} : (aucune jonction homologue)\n'); continue
        line = (f'{srcA} ↔ {srcB} : médiane {st["median"]:.1f} m | MAD {st["mad"]:.1f} m '
                f'| max {st["max"]:.1f} m  ->  c = {st["c"]} m  ({st["n_used"]} jonctions)')
        print(line)
        if st['plafonne']:
            print(f'   [PLAFOND ACTIF : calibration brute = {st["c_brut"]:.0f} m, ramenée à {c_max:.0f} m]')
        if st['outliers']:
            print(f'   non-correspondances (> c, laissées NON appariées) : '
                  f'{st["n_out"]} jonction(s), ex. {[round(v) for v in st["outliers"][:8]]} m')
        print()

    print('─' * 56)
    print('Cut-off par paire :')
    for (srcA, srcB), st in per_pair.items():
        if st['c'] is not None:
            tag = '  [plafonné]' if st['plafonne'] else ''
            print(f'  {srcA:9s} ↔ {srcB:9s} : c = {st["c"]:4d} m  '
                  f'({st["n_out"]} non-correspondance(s)){tag}')
    print(f'\n  c_global (max des c_pair) = {c_global} m')
    print('─' * 56)
    return {'per_pair': per_pair, 'c_global': c_global,
            'detail': detail, 'min_degree': min_degree}


# ============================================================
#  Execution
# ============================================================
PAIRS = list(combinations(SOURCES, 2))

if C_MANUAL is not None:
    C = C_MANUAL
    C_PAIR = {p: C_MANUAL for p in PAIRS}
    _calib = None
    print(f'Cut-off fixe manuellement : C = {C} m')
else:
    _calib = calibrate_c(
        edges, nodes, PAIRS,
        semantic_fn=semantic_label if USE_SEMANTIC else None,
        mad_k=C_MAD_K, c_max=C_MAX, min_degree=CALIB_MIN_DEGREE,
    )
    if _calib is None:
        raise ValueError('Fixer C_MANUAL avant de continuer.')

    c_global = _calib['c_global']
    C_PAIR = {p: (_calib['per_pair'][p]['c'] or c_global) for p in PAIRS}

    if C_STRATEGY == 'global':
        C = c_global
        C_PAIR = {p: c_global for p in PAIRS}
        print(f'\nStrategie « global » -> C = {C} m pour toutes les paires.')
    else:
        C = c_global
        print(f'\nStrategie « per_pair » :')
        for p, c in C_PAIR.items():
            print(f'  {p[0]} ↔ {p[1]} : c = {c} m')

print(f'\n-> β = {BETA}, γ = {GAMMA}, calibration = min(median + {C_MAD_K}*MAD, {C_MAX} m) '
      f'sur degré >= {CALIB_MIN_DEGREE}')

Noeuds retenus pour la calibration (degre >= 3) : bdtopo=985, cadastre=1054, geofla=979

=== Calibration robuste (median + 5.0*MAD) (plafond C_MAX=100 m), jonctions deg >= 3 ===

bdtopo ↔ cadastre : médiane 24.0 m | MAD 29.7 m | max 5879.9 m  ->  c = 100 m  (1189 jonctions)
   [PLAFOND ACTIF : calibration brute = 173 m, ramenée à 100 m]
   non-correspondances (> c, laissées NON appariées) : 337 jonction(s), ex. [5880, 5874, 5576, 5361, 4753, 4505, 4491, 4400] m

bdtopo ↔ geofla : médiane 14.6 m | MAD 16.0 m | max 4647.5 m  ->  c = 100 m  (987 jonctions)
   non-correspondances (> c, laissées NON appariées) : 13 jonction(s), ex. [4647, 3567, 3547, 3276, 2775, 2462, 255, 178] m

cadastre ↔ geofla : médiane 32.6 m | MAD 32.8 m | max 5887.4 m  ->  c = 100 m  (1120 jonctions)
   [PLAFOND ACTIF : calibration brute = 197 m, ramenée à 100 m]
   non-correspondances (> c, laissées NON appariées) : 303 jonction(s), ex. [5887, 5595, 5349, 4552, 4398, 4267, 4134, 3964] m

───────────────────────────

## 4. En-matching ILP — implémentation complète

Implémentation fidèle du modèle de Lei & Lei (2024), Section 3.2.

### Adaptations selon le type de données

| Paramètre | Frontières communales (actuel) | Réseaux routiers (futur) |
|---|---|---|
| `USE_SEMANTIC` | `True` (filtre INSEE) | `False` |
| Candidats arcs | même code INSEE pair | Hausdorff < c |
| Similarité arcs | Hausdorff (pour l'objectif) | Hausdorff |
| `BETA` | 3 (degré moyen = 3) | 4 (degré moyen ≈ 4) |
| `GAMMA` | 0.5 | 0.5 |
| `C` | auto (~10–30 m) | auto ou ~100 m |

### Note sur la scalabilité (grands réseaux)
Pour D031 complet (centaines de communes ou kilomètres de routes), l'ILP peut
devenir lent. Le papier (Section 4) recommande une stratégie **divide-and-conquer** :
découper la zone en blocs chevauchants, appliquer l'en-matching dans chaque bloc.
Voir Lei (2021), *Comput. Environ. Urban Syst.* 87, 101618 pour les détails.
Le paramètre `SOLVER_TIME_LIMIT` borne le temps par paire.


In [6]:
def build_candidates(srcA, srcB, nodes_by_src, edges_by_src, c,
                     use_semantic=True, semantic_fn=None,
                     restrict_nodes_A=None, restrict_nodes_B=None):
    """
    Construit les ensembles candidats N (nœuds) et E (arcs).

    v2 OPTIMISÉE pour le passage à l'échelle (département entier) — résultats
    STRICTEMENT identiques à la v1 :
      - nœuds : distances vectorisées numpy (au lieu d'une double boucle Python) ;
      - arcs  : regroupement par étiquette sémantique calculée UNE fois par côté
                (au lieu de |eA|×|eB| appels pandas .loc + semantic_fn) ;
      - repli géométrique (sans sémantique) : préfiltre STRtree avant Hausdorff.

    Retourne :
      N     : liste de (r, s) — paires de nœuds dont d(r,s) < c
      d_rs  : dict {(r,s): distance_m}
      E     : liste de (i, j) — paires d'arcs candidates
      D_ij  : dict {(i,j): hausdorff_distance_m}
    """
    from collections import defaultdict as _dd

    nA = nodes_by_src[srcA]; nB = nodes_by_src[srcB]
    eA = edges_by_src[srcA]; eB = edges_by_src[srcB]
    cA = node_coords(nA);    cB = node_coords(nB)

    # restriction préalable (mode 'junctions')
    if restrict_nodes_A is not None:
        cA = {r: xy for r, xy in cA.items() if r in restrict_nodes_A}
    if restrict_nodes_B is not None:
        cB = {s: xy for s, xy in cB.items() if s in restrict_nodes_B}

    # --- Candidats nœuds : N = {(r,s) | d(r,s) < c}, distances vectorisées ---
    N, d_rs = [], {}
    if cA and cB:
        idsA = list(cA); XA = np.array([cA[r] for r in idsA], float)
        idsB = list(cB); XB = np.array([cB[s] for s in idsB], float)
        D = np.hypot(XA[:, 0, None] - XB[None, :, 0],
                     XA[:, 1, None] - XB[None, :, 1])
        for a, b in zip(*np.where(D < c)):
            r, s = idsA[int(a)], idsB[int(b)]
            N.append((r, s))
            d_rs[(r, s)] = float(D[a, b])

    # --- Candidats arcs : E ---
    E, D_ij = [], {}
    has_geom = 'geometry' in eA.columns and 'geometry' in eB.columns
    geomA = dict(zip(eA.index, eA.geometry)) if has_geom else {}
    geomB = dict(zip(eB.index, eB.geometry)) if has_geom else {}

    if use_semantic and semantic_fn is not None:
        # étiquette sémantique calculée UNE fois par arc et par côté
        labA = _dd(list); labB = _dd(list)
        for i in eA.index:
            labA[semantic_fn(eA.loc[i])].append(i)
        for j in eB.index:
            labB[semantic_fn(eB.loc[j])].append(j)
        # seuls les arcs de MÊME étiquette sont candidats (identique au filtre v1)
        for lab, listA in labA.items():
            listB = labB.get(lab)
            if not listB:
                continue
            for i in listA:
                for j in listB:
                    D = hausdorff_dist(geomA[i], geomB[j]) if has_geom else 0.0
                    E.append((i, j))
                    D_ij[(i, j)] = D
    else:
        # repli géométrique : préfiltre spatial avant la Hausdorff (D >= c exclu)
        from shapely import STRtree
        gB = list(eB.geometry); idxB = list(eB.index)
        tree = STRtree(gB)
        for i in eA.index:
            g = geomA[i]
            for k in tree.query(g.buffer(c)):
                j = idxB[int(k)]
                D = hausdorff_dist(g, gB[int(k)])
                if D < c:
                    E.append((i, j))
                    D_ij[(i, j)] = D

    return N, d_rs, E, D_ij


def en_match_pair(srcA, srcB, nodes_by_src, edges_by_src, c,
                  beta=4.0, gamma=0.5,
                  use_semantic=True, semantic_fn=None,
                  restrict_nodes_A=None, restrict_nodes_B=None,
                  time_limit=300, verbose=False):
    """
    En-matching ILP complet (Lei & Lei 2024).

    Notation : u_nodes[(r,s)] ↔ u_rs du papier (nœuds)
               x_arcs[(i,j)]  ↔ x_ij du papier (arcs)

    L'objectif (éq. 1) pondère les NŒUDS par β et les arcs par 1+γ.
    Les 4 contraintes topologiques (éqs. 6–9) sont toutes implémentées.

    v2 OPTIMISÉE (résultats identiques) : contraintes de bijection construites en
    UN passage de E/N (au lieu d'un parcours de E par arc), extrémités d'arcs
    précalculées (plus de .loc dans la boucle des contraintes topologiques).

    Retourne :
      node_match : set {(r, s)} — nœuds appariés
      arc_match  : set {(i, j)} — arcs appariés
      obj        : valeur de l'objectif optimal
      status     : statut du solver ('Optimal', 'Time limit', ...)
    """
    from collections import defaultdict as _dd

    nA = nodes_by_src[srcA]; nB = nodes_by_src[srcB]
    eA = edges_by_src[srcA]; eB = edges_by_src[srcB]
    cA = node_coords(nA);    cB = node_coords(nB)

    # --- Construction des candidats ---
    t0 = time.time()
    N, d_rs, E, D_ij = build_candidates(
        srcA, srcB, nodes_by_src, edges_by_src, c,
        use_semantic=use_semantic, semantic_fn=semantic_fn,
        restrict_nodes_A=restrict_nodes_A, restrict_nodes_B=restrict_nodes_B
    )
    t_cand = time.time() - t0

    if not N and not E:
        return set(), set(), 0.0, 'Empty'

    # --- Constante B (éq. après éq. 1) ---
    B = compute_B(d_rs.values(), D_ij.values())

    # --- Variables binaires ---
    prob = pulp.LpProblem(f'en_{srcA}_{srcB}', pulp.LpMaximize)
    u_nodes = {(r, s): pulp.LpVariable(f'u_{r}_{s}', cat='Binary') for (r, s) in N}
    x_arcs  = {(i, j): pulp.LpVariable(f'x_{i}_{j}', cat='Binary') for (i, j) in E}

    # ── Objectif (éq. 1) ────────────────────────────────────────────────
    #  Z = Σ_(i,j)∈E [B - D_ij + γ·B]·x_ij  +  β·Σ_(r,s)∈N [B - d_rs]·u_rs
    arc_term  = pulp.lpSum((B - D_ij[(i, j)] + gamma * B) * x_arcs[(i, j)]
                           for (i, j) in E)
    node_term = beta * pulp.lpSum((B - d_rs[(r, s)]) * u_nodes[(r, s)]
                                  for (r, s) in N)
    prob += arc_term + node_term

    # ── Bijections (éqs. 2–5) — construites en UN passage de E puis N ──────
    byA, byB = _dd(list), _dd(list)
    for (i, j) in E:
        v = x_arcs[(i, j)]
        byA[i].append(v)
        byB[j].append(v)
    for i, cands in byA.items():
        prob += pulp.lpSum(cands) <= 1, f'arc_bij_A_{i}'
    for j, cands in byB.items():
        prob += pulp.lpSum(cands) <= 1, f'arc_bij_B_{j}'

    byR, byS = _dd(list), _dd(list)
    for (r, s) in N:
        v = u_nodes[(r, s)]
        byR[r].append(v)
        byS[s].append(v)
    for r, cands in byR.items():
        prob += pulp.lpSum(cands) <= 1, f'node_bij_A_{r}'
    for s, cands in byS.items():
        prob += pulp.lpSum(cands) <= 1, f'node_bij_B_{s}'

    # ── Cohérence topologique — 4 contraintes par paire d'arcs (éqs. 6–9) ──
    #  Si l'arc i est apparié à l'arc j :
    #   (6) f(i) ↔ {f(j), t(j)}   (7) f(j) ↔ {f(i), t(i)}
    #   (8) t(i) ↔ {f(j), t(j)}   (9) t(j) ↔ {f(i), t(i)}
    #  Nœuds hors-scope (mode 'junctions') : contrainte posée seulement pour les
    #  extrémités matchables (identique v1).
    endsA = {i: (int(ns), int(ne))
             for i, ns, ne in zip(eA.index, eA['node_start'], eA['node_end'])}
    endsB = {j: (int(ns), int(ne))
             for j, ns, ne in zip(eB.index, eB['node_start'], eB['node_end'])}
    matchA = restrict_nodes_A if restrict_nodes_A is not None else set(cA)
    matchB = restrict_nodes_B if restrict_nodes_B is not None else set(cB)

    for idx, (i, j) in enumerate(E):
        fi, ti = endsA[i]
        fj, tj = endsB[j]
        xij = x_arcs[(i, j)]

        def avail(pairs_list):
            return [u_nodes[p] for p in pairs_list if p in u_nodes]

        if fi in matchA:
            c6 = avail([(fi, fj), (fi, tj)])
            prob += (xij <= pulp.lpSum(c6) if c6 else xij <= 0), f'topo6_{idx}'
        if fj in matchB:
            c7 = avail([(fi, fj), (ti, fj)])
            prob += (xij <= pulp.lpSum(c7) if c7 else xij <= 0), f'topo7_{idx}'
        if ti in matchA:
            c8 = avail([(ti, fj), (ti, tj)])
            prob += (xij <= pulp.lpSum(c8) if c8 else xij <= 0), f'topo8_{idx}'
        if tj in matchB:
            c9 = avail([(fi, tj), (ti, tj)])
            prob += (xij <= pulp.lpSum(c9) if c9 else xij <= 0), f'topo9_{idx}'

    # ── Résolution ────────────────────────────────────────────────────────
    solver = pulp.PULP_CBC_CMD(msg=1 if verbose else 0, timeLimit=time_limit)
    t0 = time.time()
    prob.solve(solver)
    elapsed = time.time() - t0

    status     = pulp.LpStatus[prob.status]
    node_match = {(r, s) for (r, s), v in u_nodes.items() if (pulp.value(v) or 0) > 0.5}
    arc_match  = {(i, j) for (i, j), v in x_arcs.items()  if (pulp.value(v) or 0) > 0.5}
    obj        = pulp.value(prob.objective) or 0.0

    if verbose:
        print(f'  Candidats : {len(N)} paires nœuds, {len(E)} paires arcs '
              f'(construits en {t_cand:.1f} s)')
        print(f'  Statut : {status} | Objectif : {obj:.4f} | Solve : {elapsed:.2f} s')

    return node_match, arc_match, obj, status


## 5. Application — toutes les paires de sources

Deux modes (paramètre `MATCH_SCOPE`) :

- **`'junctions'` (rigoureux, défaut)** : seules les vraies jonctions topologiques
  (degré ≥ `MATCH_MIN_DEGREE`) sont appariées. C'est l'approche correcte — les nœuds
  de bord (degré 1) sont des artefacts de découpe de la zone test et n'ont pas
  d'homologue géographique stable. Le taux d'appariement est calculé sur les nœuds
  *matchables*, pas sur le total brut.
- **`'all'` (démonstratif)** : tous les nœuds sont appariés, bords inclus. Pratique
  pour une présentation (4/4 au lieu de 1/1 sur la zone test à 3 communes), mais les
  appariements de bord ne doivent pas entrer dans les métriques de qualité.

Sur D031 complet, la distinction s'estompe : la plupart des nœuds de bord de la zone
test deviennent de vraies jonctions avec les communes voisines.


In [7]:
results = {}  # (srcA, srcB) → (node_match, arc_match, obj, status)

# --- Portée du matching : quels nœuds sont matchables par source ---
if MATCH_SCOPE == 'junctions':
    MATCH_NODES = {src: junction_nodes(src, MATCH_MIN_DEGREE) for src in SOURCES}
    n_match = {s: len(MATCH_NODES[s]) for s in SOURCES}
    print(f'Mode RIGOUREUX (« junctions ») : seuls les nœuds de degré ≥ {MATCH_MIN_DEGREE} '
          f'sont appariés.')
    print('  Jonctions par source : ' + ', '.join(f'{s}={n_match[s]}' for s in SOURCES))
    print('  (les nœuds de bord, artefacts de découpe, sont exclus)')
    if all(v == 0 for v in n_match.values()):
        print('  ⚠  Aucune jonction de degré ≥ %d — bascule en mode « all » pour ce run.'
              % MATCH_MIN_DEGREE)
        MATCH_NODES = {src: None for src in SOURCES}
else:  # 'all'
    MATCH_NODES = {src: None for src in SOURCES}
    print('Mode DÉMONSTRATIF (« all ») : tous les nœuds sont appariés (bords inclus).')
    print('  ⚠  Les appariements de bord ne sont pas fiables — hors métriques de qualité.')

print(f'\nParamètres : β = {BETA}  γ = {GAMMA}  '
      f'(cut-off : {"manuel" if C_MANUAL is not None else C_STRATEGY})\n')
print('─' * 55)

for srcA, srcB in PAIRS:
    c_pair = C_PAIR[(srcA, srcB)]
    t0 = time.time()
    nm, am, obj, status = en_match_pair(
        srcA, srcB,
        nodes_by_src     = nodes,
        edges_by_src     = edges,
        c                = c_pair,
        beta             = BETA,
        gamma            = GAMMA,
        use_semantic     = USE_SEMANTIC,
        semantic_fn      = semantic_label if USE_SEMANTIC else None,
        restrict_nodes_A = MATCH_NODES[srcA],
        restrict_nodes_B = MATCH_NODES[srcB],
        time_limit       = SOLVER_TIME_LIMIT,
        verbose          = SOLVER_VERBOSE,
    )
    elapsed = time.time() - t0
    results[(srcA, srcB)] = (nm, am, obj, status)

    # dénominateur = nœuds matchables (pas le total brut) pour un taux honnête
    denom_n = (len(MATCH_NODES[srcA]) if MATCH_NODES[srcA] is not None
               else len(nodes[srcA]))
    pct_n = 100 * len(nm) / max(denom_n, 1)
    pct_a = 100 * len(am) / max(len(edges[srcA]), 1)
    print(f'{srcA} ↔ {srcB}  [{status}  {elapsed:.1f}s]  c = {c_pair} m')
    print(f'  nœuds : {len(nm)}/{denom_n} matchables ({pct_n:.0f}%)')
    print(f'  arcs  : {len(am)}/{len(edges[srcA])} ({pct_a:.0f}%)')
    print(f'  Z     = {obj:.4f}')
    print('─' * 55)


Mode RIGOUREUX (« junctions ») : seuls les nœuds de degré ≥ 3 sont appariés.
  Jonctions par source : bdtopo=985, cadastre=1054, geofla=979
  (les nœuds de bord, artefacts de découpe, sont exclus)

Paramètres : β = 4.0  γ = 0.5  (cut-off : per_pair)

───────────────────────────────────────────────────────
bdtopo ↔ cadastre  [Optimal  2.0s]  c = 100 m
  nœuds : 787/985 matchables (80%)
  arcs  : 987/1570 (63%)
  Z     = 27076437.7793
───────────────────────────────────────────────────────
bdtopo ↔ geofla  [Optimal  1.1s]  c = 100 m
  nœuds : 974/985 matchables (99%)
  arcs  : 1539/1570 (98%)
  Z     = 26854969.4018
───────────────────────────────────────────────────────
cadastre ↔ geofla  [Optimal  1.2s]  c = 100 m
  nœuds : 774/1054 matchables (73%)
  arcs  : 952/1691 (56%)
  Z     = 26362081.3740
───────────────────────────────────────────────────────


## 6. Résultats détaillés et cohérence transitive

La **cohérence transitive** (3-way consistency) vérifie que si BD Topo apparie
n_i à Cadastre n_j, et BD Topo apparie n_i à GÉOFLA n_k, alors Cadastre et GÉOFLA
doivent s'apparier n_j ↔ n_k. C'est une condition nécessaire pour un référentiel
de nœuds cohérent sur les 3 sources.


In [8]:
# --- Appariements de nœuds (avec distances) ---
print('=' * 60)
print('Appariements de NŒUDS (avec distances inter-sources)')
print('=' * 60)
for (srcA, srcB), (nm, am, _, _) in results.items():
    cA = node_coords(nodes[srcA]); cB = node_coords(nodes[srcB])
    print(f'\n{srcA} ↔ {srcB}:')
    for (r, s) in sorted(nm):
        d = dist_nodes(cA[r], cB[s])
        print(f'  n{r}({srcA[:3]}) ↔ n{s}({srcB[:3]}) : {d:.2f} m')
    if not nm:
        print('  (aucun appariement)')

# --- Appariements d'arcs ---
print()
print('=' * 60)
print("Appariements d'ARCS (avec labels communes)")
print('=' * 60)
for (srcA, srcB), (nm, am, _, _) in results.items():
    eA = edges[srcA]; eB = edges[srcB]
    print(f'\n{srcA} ↔ {srcB}:')
    for (i, j) in sorted(am):
        ei = eA.loc[i]; ej = eB.loc[j]
        D  = hausdorff_dist(ei.geometry, ej.geometry)              if 'geometry' in eA.columns else float('nan')
        print(f'  arc{i} [{ei.commune_a}↔{ei.commune_b}]'
              f'  ↔  arc{j} [{ej.commune_a}↔{ej.commune_b}]'
              f'  Hausdorff={D:.1f}m')
    if not am:
        print('  (aucun appariement)')

# --- Cohérence transitive 3-way ---
print()
print('=' * 60)
print('Cohérence transitive (3-way consistency)')
print('=' * 60)

src0, src1, src2 = SOURCES[0], SOURCES[1], SOURCES[2]
nm_01 = results.get((src0, src1), (set(),))[0]
nm_02 = results.get((src0, src2), (set(),))[0]
nm_12 = results.get((src1, src2), (set(),))[0]

map_01 = {r: s for (r, s) in nm_01}  # src0 → src1
map_02 = {r: k for (r, k) in nm_02}  # src0 → src2
map_12 = {(j, k) for (j, k) in nm_12}  # src1 ↔ src2

all_ok = True
for r, j in sorted(map_01.items()):
    k = map_02.get(r)
    if k is None:
        print(f'  ⚠  {src0}_n{r} → {src1}_n{j}  mais sans match {src2}')
        continue
    ok  = (j, k) in map_12
    sym = '✓' if ok else '✗'
    print(f'  {sym}  {src0}_n{r} → {src1}_n{j}, {src2}_n{k}  [{src1}↔{src2}: {sym}]')
    if not ok:
        all_ok = False

print()
if all_ok and map_01:
    print('✓  Transitivité parfaite — référentiel de nœuds cohérent sur les 3 sources.')
elif not map_01:
    print('⚠  Aucun appariement trouvé — vérifier les données et le cut-off c.')
else:
    print('✗  Incohérences détectées — voir lignes ✗ ci-dessus.')


Appariements de NŒUDS (avec distances inter-sources)

bdtopo ↔ cadastre:
  n0(bdt) ↔ n1180(cad) : 66.87 m
  n1(bdt) ↔ n941(cad) : 10.22 m
  n2(bdt) ↔ n483(cad) : 77.67 m
  n4(bdt) ↔ n1084(cad) : 8.67 m
  n6(bdt) ↔ n427(cad) : 69.96 m
  n7(bdt) ↔ n527(cad) : 73.78 m
  n8(bdt) ↔ n1140(cad) : 68.71 m
  n9(bdt) ↔ n592(cad) : 79.41 m
  n10(bdt) ↔ n298(cad) : 17.70 m
  n11(bdt) ↔ n481(cad) : 4.99 m
  n12(bdt) ↔ n1110(cad) : 5.36 m
  n13(bdt) ↔ n440(cad) : 5.00 m
  n14(bdt) ↔ n1096(cad) : 30.63 m
  n15(bdt) ↔ n753(cad) : 5.68 m
  n16(bdt) ↔ n134(cad) : 9.04 m
  n18(bdt) ↔ n268(cad) : 25.20 m
  n19(bdt) ↔ n578(cad) : 23.45 m
  n20(bdt) ↔ n1103(cad) : 0.87 m
  n21(bdt) ↔ n1168(cad) : 0.64 m
  n22(bdt) ↔ n1100(cad) : 3.55 m
  n24(bdt) ↔ n122(cad) : 93.48 m
  n25(bdt) ↔ n176(cad) : 6.32 m
  n27(bdt) ↔ n416(cad) : 24.88 m
  n29(bdt) ↔ n247(cad) : 16.52 m
  n30(bdt) ↔ n639(cad) : 3.03 m
  n32(bdt) ↔ n709(cad) : 53.56 m
  n34(bdt) ↔ n281(cad) : 12.05 m
  n35(bdt) ↔ n278(cad) : 2.25 m
  n36(bdt) ↔ n6

## 6 bis. Diagnostic — qualité de la topologie & nature des écarts

Deux vérifications complémentaires :

**(a) Cohérence nœud ↔ arcs.** Chaque nœud tombe-t-il sur l'intersection des arcs de
sa propre source ? Si oui (distance ≈ 0), la topologie construite par `build_topology.py`
est fidèle. Sinon, le nœud « flotte » → défaut de construction à corriger en amont.

**(b) Nature des écarts inter-sources.** Un nœud sain peut quand même être loin de son
homologue dans une autre source. La cause dépend du **degré** :

- **degré ≥ 3 (jonction)** : un grand écart est une **vraie divergence de données**
  (les deux sources ont digitalisé le même carrefour à des positions différentes) ;
- **degré 1 (bord)** : un grand écart est un **artefact de découpe** de la zone test
  (chaque source coupe sa frontière à un endroit différent) — **non significatif**.

C'est cette distinction qui explique le « 40 m » observé : il porte sur un nœud de
**bord** (degré 1), pas sur le point triple (degré 3, lui apparié à ~1 m).


In [9]:
from shapely.geometry import Point

TOL_M = 0.01   # un nœud est "sur ses arcs" si sa distance à chaque arc incident < TOL


def node_to_incident_arcs(src):
    """Pour chaque nœud : degré, distance aux arcs incidents, statut sain/flotte.

    Retourne une liste de dicts {node_id, degree, dmax_m, statut}.
    """
    nA = nodes[src]; eA = edges[src]
    cA = node_coords(nA)
    out = []
    for nid, xy in cA.items():
        pt = Point(xy[0], xy[1])
        dists = [pt.distance(e.geometry) for _, e in eA.iterrows()
                 if int(e.node_start) == nid or int(e.node_end) == nid]
        deg = len(dists)
        if deg == 0:
            out.append({'node_id': nid, 'degree': 0, 'dmax_m': None,
                        'statut': '⚠ orphelin'})
            continue
        dmax = max(dists)
        statut = '✓ sain' if dmax < TOL_M else f'✗ flotte ({dmax:.2f} m)'
        out.append({'node_id': nid, 'degree': deg,
                    'dmax_m': round(dmax, 4), 'statut': statut})
    return out


# ============================================================
#  (a) Cohérence nœud ↔ arcs
# ============================================================
print('=' * 64)
print('(a) Cohérence nœud ↔ arcs incidents     (tolérance %.2f m)' % TOL_M)
print('=' * 64)

any_float = False
node_degree = {}   # (src, nid) → degré, réutilisé en (b)
for src in SOURCES:
    print(f'\n{src}:')
    for row in node_to_incident_arcs(src):
        node_degree[(src, row['node_id'])] = row['degree']
        kind = 'jonction' if row['degree'] >= 3 else ('bord' if row['degree'] == 1 else 'interm.')
        d = '   —   ' if row['dmax_m'] is None else f'{row["dmax_m"]:7.4f} m'
        print(f'  n{row["node_id"]}  deg={row["degree"]} ({kind:8s})  '
              f'dist max = {d}   {row["statut"]}')
        if row['dmax_m'] is not None and row['dmax_m'] >= TOL_M:
            any_float = True

print()
if any_float:
    print('✗ Au moins un nœud ne coïncide pas avec ses arcs → vérifier build_topology.py.')
else:
    print('✓ Tous les nœuds coïncident avec leurs arcs : topologie construite fidèlement.')

# ============================================================
#  (b) Nature des écarts inter-sources (jonction vs bord)
# ============================================================
print('\n' + '=' * 64)
print('(b) Écarts inter-sources — jonctions (deg≥3) vs bords (deg 1)')
print('=' * 64)

jct_gaps, border_gaps = [], []
for (srcA, srcB), (nm, am, _, _) in results.items():
    cA = node_coords(nodes[srcA]); cB = node_coords(nodes[srcB])
    print(f'\n{srcA} ↔ {srcB}:')
    for (r, s) in sorted(nm, key=lambda p: dist_nodes(cA[p[0]], cB[p[1]]), reverse=True):
        d = dist_nodes(cA[r], cB[s])
        degA = node_degree.get((srcA, r), 0)
        degB = node_degree.get((srcB, s), 0)
        is_jct = (degA >= 3 and degB >= 3)
        tag = 'JONCTION' if is_jct else 'bord    '
        (jct_gaps if is_jct else border_gaps).append(d)
        print(f'  [{tag}] n{r}(deg{degA}) ↔ n{s}(deg{degB}) : {d:6.2f} m')

import numpy as np
print('\n' + '─' * 64)
print('Synthèse :')
if jct_gaps:
    print(f'  Jonctions (deg≥3) : médiane {np.median(jct_gaps):5.2f} m | '
          f'max {max(jct_gaps):5.2f} m  ← signal de calibration RÉEL')
else:
    print('  Jonctions (deg≥3) : aucune dans la zone test')
if border_gaps:
    print(f'  Bords (deg 1)     : médiane {np.median(border_gaps):5.2f} m | '
          f'max {max(border_gaps):5.2f} m  ← artefacts de découpe, NON significatifs')
print('─' * 64)
print('\nConclusion : pour calibrer c et juger la précision inter-sources, se fier aux')
print('JONCTIONS. Les écarts de bord (ex. ~40 m) viennent de la découpe de la zone test')
print('et disparaîtront sur D031 complet (ces points deviendront de vraies jonctions).')


(a) Cohérence nœud ↔ arcs incidents     (tolérance 0.01 m)

bdtopo:
  n0  deg=3 (jonction)  dist max =  0.0000 m   ✓ sain
  n1  deg=3 (jonction)  dist max =  0.0000 m   ✓ sain
  n2  deg=3 (jonction)  dist max =  0.0000 m   ✓ sain
  n3  deg=3 (jonction)  dist max =  0.0000 m   ✓ sain
  n4  deg=3 (jonction)  dist max =  0.0000 m   ✓ sain
  n5  deg=1 (bord    )  dist max =  0.0000 m   ✓ sain
  n6  deg=3 (jonction)  dist max =  0.0000 m   ✓ sain
  n7  deg=3 (jonction)  dist max =  0.0000 m   ✓ sain
  n8  deg=3 (jonction)  dist max =  0.0000 m   ✓ sain
  n9  deg=3 (jonction)  dist max =  0.0000 m   ✓ sain
  n10  deg=3 (jonction)  dist max =  0.0000 m   ✓ sain
  n11  deg=3 (jonction)  dist max =  0.0000 m   ✓ sain
  n12  deg=3 (jonction)  dist max =  0.0000 m   ✓ sain
  n13  deg=3 (jonction)  dist max =  0.0000 m   ✓ sain
  n14  deg=3 (jonction)  dist max =  0.0000 m   ✓ sain
  n15  deg=3 (jonction)  dist max =  0.0000 m   ✓ sain
  n16  deg=3 (jonction)  dist max =  0.0000 m   ✓ sain
  n17  

## 8. Sauvegarde

- `match_nodes.csv` : paires de nœuds appariés + coordonnées + distance  
- `match_edges.csv` : paires d'arcs appariés + label communes + distance Hausdorff  
- `en_matching_params.csv` : paramètres utilisés (traçabilité)


In [11]:
rows_nodes, rows_edges = [], []

for (srcA, srcB), (nm, am, obj, status) in results.items():
    eA = edges[srcA]; eB = edges[srcB]
    nA = nodes[srcA]; nB = nodes[srcB]
    cA = node_coords(nA); cB = node_coords(nB)
    has_geom_e = 'geometry' in eA.columns

    for (r, s) in nm:
        xr = cA[r]; xs = cB[s]
        d  = dist_nodes(xr, xs)
        rows_nodes.append({
            'source_a'  : srcA,       'source_b'  : srcB,
            'node_a'    : r,          'node_b'    : s,
            'x_a'       : round(xr[0], 3), 'y_a': round(xr[1], 3),
            'x_b'       : round(xs[0], 3), 'y_b': round(xs[1], 3),
            'distance_m': round(d, 3),
        })

    for (i, j) in am:
        ei = eA.loc[i]; ej = eB.loc[j]
        D  = round(hausdorff_dist(ei.geometry, ej.geometry), 3)              if has_geom_e else None
        rows_edges.append({
            'source_a'      : srcA, 'source_b'      : srcB,
            'edge_idx_a'    : i,    'edge_idx_b'    : j,
            'communes_a'    : f'{ei.commune_a}_{ei.commune_b}',
            'communes_b'    : f'{ej.commune_a}_{ej.commune_b}',
            'hausdorff_m'   : D,
        })

match_nodes = pd.DataFrame(rows_nodes)
match_edges = pd.DataFrame(rows_edges)

match_nodes.to_csv(f'{MATCHING_DIR}/match_nodes.csv', index=False)
match_edges.to_csv(f'{MATCHING_DIR}/match_edges.csv', index=False)

# Paramètres — traçabilité (une ligne par paire de sources)
params = pd.DataFrame([
    {
        'source_a': srcA, 'source_b': srcB,
        'c_m': C_PAIR[(srcA, srcB)],
        'beta': BETA, 'gamma': GAMMA,
        'cut_off_strategy': 'manuel' if C_MANUAL is not None else C_STRATEGY,
        'match_scope': MATCH_SCOPE,
        'calib_min_degree': CALIB_MIN_DEGREE,
        'use_semantic': USE_SEMANTIC,
        'solver_time_limit': SOLVER_TIME_LIMIT,
        'status': results[(srcA, srcB)][3],
        'objective': round(results[(srcA, srcB)][2], 4),
    }
    for (srcA, srcB) in PAIRS
])
params.to_csv(f'{MATCHING_DIR}/en_matching_params.csv', index=False)

print('Fichiers sauvegardés :')
print(f'  match_nodes.csv        — {len(match_nodes)} lignes')
print(f'  match_edges.csv        — {len(match_edges)} lignes')
print(f'  en_matching_params.csv — paramètres')
print()
print('=== match_nodes ===')
print(match_nodes.to_string(index=False))
print()
print('=== match_edges ===')
print(match_edges.to_string(index=False))


Fichiers sauvegardés :
  match_nodes.csv        — 2535 lignes
  match_edges.csv        — 3478 lignes
  en_matching_params.csv — paramètres

=== match_nodes ===
source_a source_b  node_a  node_b        x_a         y_a        x_b         y_b  distance_m
  bdtopo cadastre     854     289 545159.300 6298415.000 545139.551 6298382.411      38.106
  bdtopo cadastre     477     213 586702.200 6255919.600 586708.514 6255925.193       8.435
  bdtopo cadastre     524     145 580920.700 6278358.300 580908.409 6278358.115      12.292
  bdtopo cadastre     519     704 533765.300 6222058.800 533708.604 6222054.653      56.848
  bdtopo cadastre     365     893 575591.500 6271614.200 575585.323 6271609.830       7.566
  bdtopo cadastre     303     442 578043.800 6287994.000 578044.445 6287994.493       0.812
  bdtopo cadastre     475     396 584741.200 6273337.000 584753.963 6273355.045      22.103
  bdtopo cadastre     816     411 595660.200 6264928.400 595661.646 6264927.481       1.713
  bdtopo cad